# Evals API：图像输入

本 cookbook 演示如何将 OpenAI 的 Evals 框架用于基于图像的任务。借助 Evals API，我们将通过**采样**生成模型回答，并通过**模型评分**（LLM 作为评判者）对照图像、提示和参考答案为模型回答打分，以此评估模型针对图像和提示生成的回答。

在本示例中，我们将评估模型在以下方面的表现：
1. 针对用户关于图像的提示**生成恰当的回答**
3. 与代表高质量回答的**参考答案保持一致**

## 安装依赖与设置

In [1]:
# Install required packages
!pip install openai datasets pandas --quiet

In [ ]:
# Import libraries
from datasets import load_dataset
from openai import OpenAI
import os
import json
import time
import pandas as pd

## 数据集准备

我们使用托管在 Hugging Face 上的 [VibeEval](https://huggingface.co/datasets/RekaAI/VibeEval) 数据集。它包含一系列用户提示、配套图像和参考答案数据。首先加载数据集。

In [3]:
dataset = load_dataset("RekaAI/VibeEval")

我们提取相关字段，并将其整理成类似 json 的格式，作为数据源传入 Evals API。输入图像数据可以是 Web URL，也可以是 base64 编码的字符串。这里我们使用数据集提供的 Web URL。 

In [ ]:
evals_data_source = []

# select the first 3 examples in the dataset to use for this cookbook
for example in dataset["test"].select(range(3)):
    evals_data_source.append({
        "item": {
            "media_url": example["media_url"], # image web URL
            "reference": example["reference"], # reference answer
            "prompt": example["prompt"] # prompt
        }
    })

如果打印数据源列表，每一项的形式应类似于：

```python
{
  "item": {
    "media_url": "https://storage.googleapis.com/reka-annotate.appspot.com/vibe-eval/difficulty-normal_food1_7e5c2cb9c8200d70.jpg"
    "reference": "This appears to be a classic Margherita pizza, which has the following ingredients..."
    "prompt": "What ingredients do I need to make this?"
  }
}
```

## 评估配置

现在我们已经有了数据源和任务，接下来创建评估。OpenAI Evals API 文档请参阅 [API 文档](https://platform.openai.com/docs/evals/overview)。


In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

评估分为两部分："Eval" 和 "Run"。在 "Eval" 中，我们定义数据的预期结构和测试标准（评分器）。

### 数据源配置

根据我们整理的数据，数据源配置如下：

In [28]:
data_source_config = {
    "type": "custom",
    "item_schema": {
        "type": "object",
        "properties": {
          "media_url": { "type": "string" },
          "reference": { "type": "string" },
          "prompt": { "type": "string" }
        },
        "required": ["media_url", "reference", "prompt"]
      },
    "include_sample_schema": True, # enables sampling
}

### 测试标准

对于测试标准，我们设置评分器配置。在本示例中，它是一个模型评分器，接收图像、参考答案和采样得到的模型回答（位于 `sample` 命名空间中），然后根据模型回答与参考答案的接近程度以及它对对话的总体适用性，输出一个 0 到 1 之间的分数。有关模型评分器的更多信息，请参阅 [API 评分器文档](https://platform.openai.com/docs/api-reference/graders)。 

数据和评分器都设置得当，是有效评估的关键。因此，你很可能需要迭代地完善评分器的提示。 

**注意**：图像 url 字段/模板需要放在 input image 对象中，才能被解析为图像。否则，图像会被解析为文本字符串。 

In [29]:
grader_config = {
	    "type": "score_model",
        "name": "Score Model Grader",
        "input":[
            {
                "role": "system",
		        "content": "You are an expert grader. Judge how well the model response suits the image and prompt as well as matches the meaniing of the reference answer. Output a score of 1 if great. If it's somewhat compatible, output a score around 0.5. Otherwise, give a score of 0."
	        },
	        {
		        "role": "user",
		        "content": [{ "type": "input_text", "text": "Prompt: {{ item.prompt }}."},
							{ "type": "input_image", "image_url": "{{ item.media_url }}", "detail": "auto" },
							{ "type": "input_text", "text": "Reference answer: {{ item.reference }}. Model response: {{ sample.output_text }}."}
				]
	        }
		],
		"pass_threshold": 0.9,
	    "range": [0, 1],
	    "model": "o4-mini" # model for grading; check that the model you use supports image inputs
	}

现在创建 eval 对象。

In [ ]:
eval_object = client.evals.create(
        name="Image Grading",
        data_source_config=data_source_config,
        testing_criteria=[grader_config],
    )

## 评估运行

创建运行时，我们传入 eval 对象 id、数据源（即我们之前整理的数据），以及用于采样生成模型回答的聊天消息输入。注意，EvalsAPI 还支持将包含图像的已存储 completions 和 responses 作为数据源。更多信息请参阅[补充信息：日志数据源](#补充信息日志数据源)一节。

以下是本示例将使用的采样消息输入。

In [31]:
sampling_messages = [{
    "role": "user",
    "type": "message",
    "content": {
        "type": "input_text",
        "text": "{{ item.prompt }}"
      }
  },
  {
    "role": "user",
    "type": "message",
    "content": {
        "type": "input_image",
        "image_url": "{{ item.media_url }}",
        "detail": "auto"
    }
  }]

现在启动一次评估运行。

In [32]:
eval_run = client.evals.runs.create(
        name="Image Input Eval Run",
        eval_id=eval_object.id,
        data_source={
            "type": "responses", # sample using responses API
            "source": {
                "type": "file_content",
                "content": evals_data_source
            },
            "model": "gpt-4o-mini", # model used to generate the response; check that the model you use supports image inputs
            "input_messages": {
                "type": "template", 
                "template": sampling_messages}
        }
    )

## 轮询并展示结果

运行结束后，我们就可以查看结果。你也可以在组织的 OpenAI evals 仪表板中查看进度和结果。 

In [46]:
while True:
    run = client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    if run.status == "completed" or run.status == "failed": # check if the run is finished
        output_items = list(client.evals.runs.output_items.list(
            run_id=run.id, eval_id=eval_object.id
        ))
        df = pd.DataFrame({
                "prompt": [item.datasource_item["prompt"] for item in output_items],
                "reference": [item.datasource_item["reference"] for item in output_items],
                "model_response": [item.sample.output[0].content for item in output_items],
                "score": [item.results[0].score for item in output_items],
                "passed": [item.results[0].passed for item in output_items],
                "grading_results": [item.results[0].sample["output"][0]["content"] for item in output_items]
            })
        display(df)
        break
    time.sleep(5)

,prompt,reference,model_response,score,passed,grading_results
0,Please provide latex code to replicate this table,Below is the latex code for your table:\n```te...,Certainly! Below is the LaTeX code to replicat...,0.5,False,"{""steps"":[{""description"":""Interpreted prompt r..."
1,What ingredients do I need to make this?,"This appears to be a classic Margherita pizza,...",To make a classic Margherita pizza similar to ...,1.0,True,"{""steps"":[{""description"":""Compare model's ingr..."
2,Is this safe for a vegan to eat?,"Based on the image, this dish appears to be a ...","To determine if the dish is safe for a vegan, ...",1.0,True,"{""steps"":[{""description"":""Compare model respon..."


### 查看单个输出项

要查看完整的输出项，可以按如下方式操作。输出项的结构在 API 文档中有说明，请参阅[这里](https://platform.openai.com/docs/api-reference/evals/run-output-item-object)。

In [47]:
first_item = output_items[0]

print(json.dumps(dict(first_item), indent=2, default=str))

{
  "id": "outputitem_687833f102ec8191a6e53a5461b970c2",
  "created_at": 1752708081,
  "datasource_item": {
    "prompt": "Please provide latex code to replicate this table",
    "media_url": "https://storage.googleapis.com/reka-annotate.appspot.com/vibe-eval/difficulty-normal_table0_b312eea68bcd0de6.png",
    "reference": "Below is the latex code for your table:\n```tex\n\\begin{table}\n\\begin{tabular}{c c c c} \\hline  & \\(S2\\) & Expert & Layman & PoelM \\\\ \\cline{2-4} \\(S1\\) & Expert & \u2013 & 54.0 & 62.7 \\\\  & Layman & 46.0 & \u2013 & 60.7 \\\\  &,PoelM,LM,LM,LM,LM,LM,,L,M,,L,M,,L,M,,L,M,,,\u2013&39.3 \\\\\n[-1ex] \\end{tabular}\n\\end{table}\n```."
  },
  "datasource_item_id": 1,
  "eval_id": "eval_687833d68e888191bc4bd8b965368f22",
  "object": "eval.run.output_item",
  "results": [
    {
      "name": "Score Model Grader-73fe48a0-8090-46eb-aa8e-d426ad074eb3",
      "sample": {
        "input": [
          {
            "role": "system",
            "content": "You are a

## 补充信息：日志数据源

如前所述，EvalsAPI 支持将包含图像的日志（即已存储的 completions 或 responses）作为数据源。要使用此功能，请按如下方式修改评估配置： 

创建 Eval
  - 设置 `data_source_config = { "type": "logs" }`
  - 修改 `grader_config` 中的模板，使用 `{{item.input}}` 和/或 `{{sample.output_text}}`，分别表示日志的输入和输出

创建 Eval Run
  - 在 `data_source` 字段中指定过滤条件，用于获取本次评估运行对应的日志（更多信息请参阅[文档](https://platform.openai.com/docs/api-reference/evals/createRun)）

## 结论

在本 cookbook 中，我们介绍了使用 OpenAI Evals API 评估基于图像的任务的工作流。通过在采样和模型评分中都使用图像输入功能，我们简化了该任务的评估流程。

我们期待看到你将其扩展到自己的图像类用例中，无论是 OCR 准确率、图像生成评分，还是更多场景！